# Load Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import torch


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load Data

In [4]:
ticker = 'AAPL'
# ticker = "GOOGL"
# ticker = "^IXIC"
# ticker = "^GSPC"

# 1. Prepare Data
df = yf.download(ticker, start='2013-01-01')['Close']
df = df.asfreq('B')


[*********************100%***********************]  1 of 1 completed


# Data Partitioning

In [5]:
train_size = int(0.8 * len(df))
train = df[:train_size]
test = df[train_size:]

# Set input window and forcast horizon length 
window_size = 35  
forecast_horizon = 5  
context_length = window_size - forecast_horizon  

print(f"Window Size: {window_size}")
print(f"Forecast Horizon: {forecast_horizon}")
print(f"Context Length: {context_length}")

all_predictions = []
all_actuals = []

num_windows = len(test) - forecast_horizon + 1

def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    return 100 * np.mean(np.where(denominator == 0, 0, diff / denominator))

Window Size: 35
Forecast Horizon: 5
Context Length: 30


# Training

In [6]:
for i in range(num_windows):
    try:
        # Get the sliding window of data
        all_data = pd.concat([train, test.iloc[:i]])
        
        if len(all_data) >= context_length:
            window_data = all_data.iloc[-context_length:]
        else:
            window_data = all_data
        
        # Skip if window is too small
        if len(window_data) < 10:
            continue
        
        # Fit ARIMA on the window
        model = SARIMAX(window_data, order=(2,1,2))
        results = model.fit(disp=False, maxiter=200, method='lbfgs')
        
        # Forecast the next 'forecast_horizon' steps
        forecast = results.forecast(steps=forecast_horizon)
        
        # Get actual values for this forecast horizon
        actual = test.iloc[i:i+forecast_horizon].values
        
        # Check if forecast has valid values
        if not np.isnan(forecast.values).any() and len(forecast) == forecast_horizon:
            all_predictions.append(forecast.values)
            all_actuals.append(actual)
        
        if (i+1) % 20 == 0:
            print(f"Processed {i+1}/{num_windows} windows")
            
    except Exception as e:
        # Skip windows that fail to fit
        continue

print(f"\nSuccessfully created {len(all_predictions)} forecasts out of {num_windows} windows")

# 4. Calculate Metrics only if we have valid predictions
if len(all_predictions) > 0:
    # Flatten all predictions and actuals
    y_pred_flat = np.concatenate(all_predictions)
    y_true_flat = np.concatenate(all_actuals)
    
    # Ensure arrays are 1D and flatten if needed
    y_pred_flat = y_pred_flat.flatten()
    y_true_flat = y_true_flat.flatten()
    
    # Remove any remaining NaN values (safety check)
    valid_indices = ~(np.isnan(y_pred_flat) | np.isnan(y_true_flat))
    y_pred_flat = y_pred_flat[valid_indices]
    y_true_flat = y_true_flat[valid_indices]
    
    print(f"Total valid predictions: {len(y_pred_flat)}")
    
    if len(y_pred_flat) > 0:
        rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
        mae = mean_absolute_error(y_true_flat, y_pred_flat)
        mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
        r2 = r2_score(y_true_flat, y_pred_flat)
        smape_value = smape(y_true_flat, y_pred_flat)

        
        print("\n" + "="*60)
        print("PERFORMANCE METRICS (Multi-step Forecast)")
        print("="*60)
        print(f'RMSE: {rmse:.4f}')
        print(f'MAE:  {mae:.4f}')
        print(f'MAPE: {mape:.4f} ({mape*100:.2f}%)')
        print(f'SMAPE: {smape_value:.4f}%')
        print(f'R²:   {r2:.4f}')
        print("="*60)
        
        

    else:
        print("ERROR: All predictions were filtered out as invalid!")
else:
    print("ERROR: No valid predictions were generated!")

Processed 20/681 windows
Processed 40/681 windows
Processed 60/681 windows
Processed 80/681 windows
Processed 100/681 windows
Processed 120/681 windows
Processed 140/681 windows
Processed 160/681 windows
Processed 180/681 windows
Processed 200/681 windows
Processed 220/681 windows
Processed 240/681 windows
Processed 260/681 windows
Processed 280/681 windows
Processed 300/681 windows
Processed 320/681 windows
Processed 340/681 windows
Processed 360/681 windows
Processed 380/681 windows
Processed 400/681 windows
Processed 420/681 windows
Processed 440/681 windows
Processed 460/681 windows
Processed 480/681 windows
Processed 500/681 windows
Processed 520/681 windows
Processed 540/681 windows
Processed 560/681 windows
Processed 580/681 windows
Processed 600/681 windows
Processed 620/681 windows
Processed 640/681 windows
Processed 660/681 windows
Processed 680/681 windows

Successfully created 680 forecasts out of 681 windows
Total valid predictions: 3268

PERFORMANCE METRICS (Multi-step Fo